<a href="https://colab.research.google.com/github/YRMESHRAM/Machine-Vision-Practical/blob/main/MV_P7_D4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
# Download a sample traffic video
!wget -O traffic_sample.mp4 https://github.com/intel-iot-devkit/sample-videos/raw/master/car-detection.mp4 -q

import cv2
import numpy as np
import matplotlib.pyplot as plt

print("Dependencies imported and sample video downloaded successfully!")

Dependencies imported and sample video downloaded successfully!


In [26]:
# Install ffmpeg for HTML video playback in Colab
!apt-get update && apt-get install -y ffmpeg

import cv2
import numpy as np
import urllib.request
from collections import OrderedDict, deque
from IPython.display import HTML
from base64 import b64encode

# Download sample traffic video from public repository
video_url = "https://github.com/intel-iot-devkit/sample-videos/raw/master/car-detection.mp4"
urllib.request.urlretrieve(video_url, "traffic_input.mp4")
print("Sample traffic video downloaded successfully as 'traffic_input.mp4'.")

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [102 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,145 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,155 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.6 MB]
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:14 http://

In [27]:
class VehicleTracker:
    def __init__(self, maxDisappeared=15, maxDistance=60, maxHistory=30):
        self.nextObjectID = 0
        self.objects = OrderedDict()       # ID -> (x, y) centroid
        self.disappeared = OrderedDict()   # ID -> frame count missing
        self.history = OrderedDict()       # ID -> deque of previous (x, y) points
        self.maxDisappeared = maxDisappeared
        self.maxDistance = maxDistance
        self.maxHistory = maxHistory

    def register(self, centroid):
        self.objects[self.nextObjectID] = centroid
        self.disappeared[self.nextObjectID] = 0
        self.history[self.nextObjectID] = deque(maxlen=self.maxHistory)
        self.history[self.nextObjectID].append(centroid)
        self.nextObjectID += 1

    def deregister(self, objectID):
        del self.objects[objectID]
        del self.disappeared[objectID]
        del self.history[objectID]

    def update(self, rects):
        if len(rects) == 0:
            for objectID in list(self.disappeared.keys()):
                self.disappeared[objectID] += 1
                if self.disappeared[objectID] > self.maxDisappeared:
                    self.deregister(objectID)
            return self.objects, self.history

        # Compute centroids for current frame detections
        inputCentroids = np.zeros((len(rects), 2), dtype="int")
        for (i, (startX, startY, endX, endY)) in enumerate(rects):
            cX = int((startX + endX) / 2.0)
            cY = int((startY + endY) / 2.0)
            inputCentroids[i] = (cX, cY)

        if len(self.objects) == 0:
            for i in range(0, len(inputCentroids)):
                self.register(inputCentroids[i])
        else:
            objectIDs = list(self.objects.keys())
            objectCentroids = list(self.objects.values())

            # Distance matrix between existing objects and new centroids
            D = np.linalg.norm(np.array(objectCentroids)[:, np.newaxis] - inputCentroids, axis=2)
            rows = D.min(axis=1).argsort()
            cols = D.argmin(axis=1)[rows]

            usedRows, usedCols = set(), set()

            for (row, col) in zip(rows, cols):
                if row in usedRows or col in usedCols:
                    continue
                if D[row, col] > self.maxDistance:
                    continue

                objectID = objectIDs[row]
                self.objects[objectID] = inputCentroids[col]
                self.history[objectID].append(inputCentroids[col])
                self.disappeared[objectID] = 0

                usedRows.add(row)
                usedCols.add(col)

            unusedRows = set(range(0, D.shape[0])).difference(usedRows)
            unusedCols = set(range(0, D.shape[1])).difference(usedCols)

            if D.shape[0] >= D.shape[1]:
                for row in unusedRows:
                    objectID = objectIDs[row]
                    self.disappeared[objectID] += 1
                    if self.disappeared[objectID] > self.maxDisappeared:
                        self.deregister(objectID)
            else:
                for col in unusedCols:
                    self.register(inputCentroids[col])

        return self.objects, self.history

In [28]:
# Input/Output Video setup
cap = cv2.VideoCapture("traffic_input.mp4")
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = cap.get(cv2.CAP_PROP_FPS)

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('output_raw.mp4', fourcc, fps, (width, height))

# 1. Background Subtractor initialization
bg_subtractor = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=50, detectShadows=True)

# 2. Vehicle Tracker initialization
tracker = VehicleTracker(maxDisappeared=10, maxDistance=50)

# Read initial frame for Optical Flow
ret, prev_frame = cap.read()
prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

print("Processing video frames...")
frame_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # --- A. BACKGROUND SUBTRACTION & NOISE REDUCTION ---
    fg_mask = bg_subtractor.apply(frame)
    _, fg_mask = cv2.threshold(fg_mask, 200, 255, cv2.THRESH_BINARY) # Remove shadow (gray) values
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_OPEN, kernel)
    fg_mask = cv2.dilate(fg_mask, kernel, iterations=2)

    # --- B. CONTOUR DETECTION (Bounding Boxes) ---
    contours, _ = cv2.findContours(fg_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    rects = []
    for cnt in contours:
        if cv2.contourArea(cnt) > 400: # Filter small noisy contours
            x, y, w, h = cv2.boundingRect(cnt)
            rects.append((x, y, x + w, y + h))
            cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 0, 0), 2) # Blue box

    # --- C. OBJECT TRACKING & TRAJECTORY DRAWING ---
    objects, history = tracker.update(rects)

    for (objectID, centroid) in objects.items():
        # Draw Trajectory tail (red line)
        pts = history[objectID]
        for i in range(1, len(pts)):
            if pts[i - 1] is None or pts[i] is None:
                continue
            thickness = int(np.sqrt(32 / float(i + 1)) * 1.5)
            cv2.line(frame, tuple(pts[i - 1]), tuple(pts[i]), (0, 0, 255), thickness)

        # Draw vehicle ID tag
        cv2.circle(frame, (centroid[0], centroid[1]), 4, (0, 255, 0), -1)
        cv2.putText(frame, f"ID {objectID}", (centroid[0] - 10, centroid[1] - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # --- D. DENSE OPTICAL FLOW (Farneback Motion Field) ---
    flow = cv2.calcOpticalFlowFarneback(prev_gray, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)

    # Plot optical flow vectors in a grid pattern
    step = 24
    for y in range(step // 2, height, step):
        for x in range(step // 2, width, step):
            fx, fy = flow[y, x]
            mag = np.hypot(fx, fy)
            if mag > 1.5: # Render vectors for significant movement only
                cv2.line(frame, (x, y), (int(x + fx), int(y + fy)), (0, 255, 255), 1)
                cv2.circle(frame, (x, y), 1, (0, 255, 255), -1)

    # Frame Overlay HUD
    cv2.putText(frame, f"Active Tracking Count: {len(objects)}", (20, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    out.write(frame)
    prev_gray = gray.copy()
    frame_count += 1

cap.release()
out.release()
print(f"Done! Processed {frame_count} frames. Output saved to 'output_raw.mp4'.")

Processing video frames...
Done! Processed 376 frames. Output saved to 'output_raw.mp4'.


In [29]:
# Convert output video to H.264 format for HTML5 browser rendering
!ffmpeg -y -i output_raw.mp4 -vcodec libx264 -f mp4 vehicle_analysis_output.mp4 -loglevel quiet

# Render HTML5 Video player in Colab
mp4_data = open('vehicle_analysis_output.mp4', 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4_data).decode()

HTML(f"""
<video width=700 controls>
      <source src="{data_url}" type="video/mp4">
</video>
""")